In [1]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk > /dev/null
!update-alternatives --install /usr/bin/java java /usr/lib/jvm/java-17-openjdk-amd64/bin/java 1
!update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java

!pip install -q transformers torch sentencepiece nltk
!pip install -q git+https://github.com/PrithivirajDamodaran/Parrot_Paraphraser
!pip install -q language-tool-python




W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/java to provide /usr/bin/java (java) in auto mode
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/java to provide /usr/bin/java (java) in manual mode
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [4]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
import nltk
nltk.download('punkt')

import torch
from parrot import Parrot
import language_tool_python
from nltk.tokenize import sent_tokenize
from concurrent.futures import ThreadPoolExecutor

use_gpu = torch.cuda.is_available()
print(f"✅ GPU available: {use_gpu}")

parrot = Parrot(model_tag="prithivida/parrot_paraphraser_on_T5", use_gpu=use_gpu)
tool = language_tool_python.LanguageTool('en-US')

def split_into_chunks(text, max_sentences=100):
    sentences = sent_tokenize(text)
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = sentences[i:i+max_sentences]
        chunks.append(chunk)
    return chunks

def batch_paraphrase(sentences):
    results = []
    for sent in sentences:
        paras = parrot.augment(input_phrase=sent, use_gpu=use_gpu, max_return_phrases=5, do_diverse=True)
        if paras and len(paras) > 0:
            best_para = paras[0][0]
        else:
            best_para = sent
        results.append(best_para)
    return results


def correct_sentences(sentences):
    # Ensure all elements are strings
    safe_sentences = [s if isinstance(s, str) else " ".join(s) for s in sentences]

    with ThreadPoolExecutor(max_workers=8) as executor:
        corrected = list(executor.map(tool.correct, safe_sentences))

    return corrected


def paraphrase_and_correct(chunk_sentences):
    # paraphrase chunk sentences in batch
    paraphrased = batch_paraphrase(chunk_sentences)
    # grammar correction in parallel
    corrected = correct_sentences(paraphrased)
    return " ".join(corrected)

def process_paragraphs(paragraphs_list):
    output = []
    for idx, para in enumerate(paragraphs_list, start=1):
        print(f"Processing paragraph {idx}/{len(paragraphs_list)}...")
        chunks = split_into_chunks(para, max_sentences=100)
        para_result = ""
        for cidx, chunk_sentences in enumerate(chunks, start=1):
            print(f" - Chunk {cidx}/{len(chunks)}")
            para_result += paraphrase_and_correct(chunk_sentences) + "\n\n"
        output.append(para_result.strip())
    return output


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/usr/local/lib/python3.11/dist-packages/transformers/models/auto/tokenization_auto.py:898: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


✅ GPU available: True


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py:476: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output

input_box = widgets.Textarea(
    value='',
    placeholder='Paste your paragraphs here...',
    description='Input Text:',
    layout=widgets.Layout(width='100%', height='200px'),
    style={'description_width': 'initial'}
)

output_box = widgets.Output(layout=widgets.Layout(border='1px solid black', width='100%', height='300px', overflow='auto'))

button = widgets.Button(description="Process Text", button_style='primary')

def on_button_clicked(b):
    with output_box:
        clear_output()
        text = input_box.value.strip()
        if not text:
            print("Please enter some text to process.")
            return

        # Split paragraphs by double newlines (empty lines)
        paragraphs = [p.strip().replace('\n', ' ') for p in text.split('\n\n') if p.strip()]
        results = process_paragraphs(paragraphs)

        print("\n\n=== Humanized High-Level Academic Output ===\n")
        for i, res in enumerate(results, 1):
            print(f"Paragraph {i}:\n{res}\n{'-'*60}\n")

button.on_click(on_button_clicked)

display(input_box, button, output_box)


Textarea(value='', description='Input Text:', layout=Layout(height='200px', width='100%'), placeholder='Paste …

Button(button_style='primary', description='Process Text', style=ButtonStyle())

Output(layout=Layout(border='1px solid black', height='300px', overflow='auto', width='100%'))